# Analog Holidays - 38h Offset Forecast + Cluster Filter

Thin orchestration notebook for running `AnalogSpecialDays` from the hourly wide holiday audit CSV.

This variant forecasts a 38-hour window that starts 14 hours before the holiday midnight, so the operational forecast is ready before the holiday begins.

The loading, normalization, forecasting, and plotting logic lives in `analog/analog_holidays.py`. This notebook only defines parameters and calls the plotting helpers.

The CSV export only preserves holiday flags, so this workflow targets holiday analogs and restricts the analog bank to the selector cluster `F/G/H` assigned to each target in `holiday_selector_features.csv`.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module = importlib.reload(analog_holidays_module)

from analog_holidays.analog.analog_holidays import (
    build_analog_ranking_table,
    build_run_summary,
    plot_analog_pair_sequences,
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    plot_forecast_diagnostics,
    plot_ranked_analog_profiles,
    prepare_audit_working_copy,
    run_analog_holidays,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Parameters

Adjust the series, target date, and analog hyperparameters for the holiday-only CSV source.

- SOURCE_PATH: path to the holiday CSV file consumed by the notebook.
- UNIQUE_ID: target series used to build analogs and generate the forecast.
- TARGET_DATE: holiday date whose operational forecast window should be estimated.
- FORECAST_START_OFFSET_HOURS: how many hours before the holiday midnight the forecast window starts.
- SEASON_LENGTH: hourly profile length to model, in hours. For this workflow it should match `FORECAST_START_OFFSET_HOURS + 24`.
- SPECIAL_LABELS: labels that define which days are treated as special candidates when selecting analogs.
- K: number of special neighbors kept after ranking X against Y by similarity; None uses every filtered candidate.
- TYPEDIST: metric used to rank holiday candidates against Y; supports pearson, euclidian, and dtw.
- TYPEREG: regressor type used in the analog reconstruction step; supports PCR, PLS, RidgeReg, LassoReg, RF, OLSstep, and LGBM when `lightgbm` is installed.
- SCALE_METHOD: optional preprocessing transform applied before neighbor selection and regression, then inverted back to the original demand scale for the final forecast. Supported values: `None`, `standard`, `minmax`.
- N_COMPONENTS: number of components for dimensionality-reduction methods such as PCR or PLS; ignored by tree-based regressors.
- REGRESSOR_PARAMS: optional dict of model-specific constructor hyperparameters, mainly for RF or LGBM when running a manual configuration outside Optuna.
- LEVELS: prediction interval levels to compute for the forecast.
- MIN_SPECIAL_POINTS: minimum number of hours flagged as special inside a candidate block.
- MIN_EVENT_GAP: minimum separation between consecutive special events to avoid overly overlapping candidates.
- MAX_EVENTS: maximum number of special events used as the analog bank; None uses all available events.
- SELECTOR_FEATURES_PATH / CLUSTER_COLUMN / MATCH_TARGET_CLUSTER: selector-cluster filter that keeps only historical analogs in the same target cluster (`F`, `G`, or `H`) defined in `holiday_selector_features.csv`.
- MAX_PLOTTED_ANALOGS: maximum number of analogs shown in the comparative plots.

In [ ]:
import shutil
from datetime import datetime

# Original CSV — never modified by this notebook.
_ORIGINAL_SOURCE = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_mx.csv'

# Create a fresh timestamped working copy on every notebook launch.
_timestamp = datetime.now().strftime('%Y_%m_%d_%H_%M')
SOURCE_PATH = _ORIGINAL_SOURCE.with_name(f'holiday_demand_mx_{_timestamp}.csv')
shutil.copy2(_ORIGINAL_SOURCE, SOURCE_PATH)
print(f'Working copy: {SOURCE_PATH.name}')

UNIQUE_IDS = analog_holidays_module.get_available_unique_ids(SOURCE_PATH)
if not UNIQUE_IDS:
    raise ValueError(f'No unique_id values were found in {SOURCE_PATH.name}.')

UNIQUE_ID = 'SEN_demand_SIN'
if UNIQUE_ID not in UNIQUE_IDS:
    UNIQUE_ID = UNIQUE_IDS[0]

print(f'Series to forecast: {len(UNIQUE_IDS)}')
print(', '.join(UNIQUE_IDS))
print(f'Detail view series: {UNIQUE_ID}')

In [ ]:
SPECIAL_LABELS = ('holiday',)
FORECAST_START_OFFSET_HOURS = 14
SEASON_LENGTH = 38  # 14 h pre-holiday + 24 h holiday
K = 1000
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
SCALE_METHOD = None
OPTUNA_SCALE_METHOD_CHOICES = [None, 'standard', 'minmax']
N_COMPONENTS = 3
REGRESSOR_PARAMS = {}
LEVELS = [80, 95]
MIN_SPECIAL_POINTS = 24  # require the 24 holiday hours inside each 38-h candidate window
MIN_EVENT_GAP = 24
MAX_EVENTS = None
MAX_PLOTTED_ANALOGS = 10

SELECTOR_FEATURES_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_selector_features.csv'
CLUSTER_COLUMN = 'analog_cluster'
MATCH_TARGET_CLUSTER = True

DATE_END = '2024-01-01'  # only dates strictly earlier than this cutoff are included in the study
OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 900
OPTUNA_MAX_EVAL_DATES = 12
OPTUNA_RANDOM_SEED = 42

In [ ]:
if OPTUNA_SCALE_METHOD_CHOICES is not None:
    valid_scale_methods = {None, 'standard', 'minmax'}
    invalid_scale_methods = [
        value for value in OPTUNA_SCALE_METHOD_CHOICES
        if value not in valid_scale_methods
    ]
    if invalid_scale_methods:
        raise ValueError(
            f'Unsupported OPTUNA_SCALE_METHOD_CHOICES: {invalid_scale_methods}'
        )

print(f'SCALE_METHOD fixed fallback: {SCALE_METHOD}')
print(f'OPTUNA scale choices: {OPTUNA_SCALE_METHOD_CHOICES}')

In [ ]:
TARGET_DATES_2025 = [
    ('2025-01-01', "New Year's Day"),
    ('2025-02-03', 'Constitution Day'),
    ('2025-03-17', "Benito Juarez's Birthday"),
    ('2025-04-17', 'Maundy Thursday'),
    ('2025-04-18', 'Good Friday'),
    ('2025-04-19', 'Holy Saturday'),
    ('2025-05-01', 'Labor Day'),
    ('2025-09-16', 'Independence Day'),
    ('2025-11-17', 'Mexican Revolution Day'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    ('2025-12-31', "New Year's Eve"),
    # ===== 2026 =====
    ('2026-01-01', "New Year's Day"),    
    ('2026-02-02', 'Constitution Day'),
    ('2026-03-16', "Benito Juarez's Birthday"),
    ('2026-04-02', 'Maundy Thursday'),
    ('2026-04-03', 'Good Friday'),
    ('2026-04-04', 'Holy Saturday'),
    ('2026-05-01', 'Labor Day'),
    # ('2026-09-15', 'Independence Eve'),
    # ('2026-09-16', 'Independence Day'),
    # ('2026-11-16', 'Mexican Revolution Day'),
    # ('2026-12-24', 'Christmas Eve'),
    # ('2026-12-25', 'Christmas Day'),
    # ('2026-12-31', "New Year's Eve"),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

from matplotlib.backends.backend_pdf import PdfPages

RESULTS_DIR = PROJECT_ROOT / 'analog_holidays' / 'Holiday_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MULTIPAGE_PDF_PATH = RESULTS_DIR / 'all_holiday_graphs.pdf'
PDF_FIGURES = {}

def _pdf_safe_name(value):
    safe = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))
    while '__' in safe:
        safe = safe.replace('__', '_')
    return safe.strip('_') or 'figure'

def export_figure_pdf(fig_obj, stem):
    safe_stem = _pdf_safe_name(stem)
    single_path = RESULTS_DIR / f'{safe_stem}.pdf'
    fig_obj.savefig(single_path, format='pdf', bbox_inches='tight')
    PDF_FIGURES[safe_stem] = fig_obj
    with PdfPages(MULTIPAGE_PDF_PATH) as pdf:
        for saved_fig in PDF_FIGURES.values():
            pdf.savefig(saved_fig, bbox_inches='tight')
    print(f'Saved PDF: {single_path}')
    print(f'Updated multipage PDF: {MULTIPAGE_PDF_PATH}')
    return single_path, MULTIPAGE_PDF_PATH

print(f'PDF output folder: {RESULTS_DIR}')

In [ ]:
selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
target_ts = pd.Timestamp(TARGET_DATE).normalize()

target_cluster_df = selector_features_df.loc[
    selector_features_df['date'] == target_ts,
    ['date', 'holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
]

if target_cluster_df.empty:
    raise ValueError(
        f'No selector cluster was found for TARGET_DATE={target_ts.date()} in {SELECTOR_FEATURES_PATH.name}.'
    )

TARGET_ANALOG_CLUSTER = target_cluster_df.iloc[0][CLUSTER_COLUMN]
eligible_cluster_analogs_df = selector_features_df.loc[
    (selector_features_df[CLUSTER_COLUMN] == TARGET_ANALOG_CLUSTER)
    & (selector_features_df['date'] < target_ts),
    ['date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
] .sort_values('date').reset_index(drop=True)

print(
    f'TARGET_DATE={target_ts.date()} -> analog_cluster={TARGET_ANALOG_CLUSTER} '
    f'| eligible historical analog dates={len(eligible_cluster_analogs_df)}'
 )
display(target_cluster_df)
eligible_cluster_analogs_df.tail(20)

In [ ]:
rolling_target_items = [
    (pd.Timestamp(target_date).date().isoformat(), holiday_label)
    for target_date, holiday_label in TARGET_DATES_2025
]

selector_cluster_lookup = (
    selector_features_df
    .dropna(subset=[CLUSTER_COLUMN])
    .drop_duplicates(subset=['date'], keep='last')
    .set_index('date')[CLUSTER_COLUMN]
    .to_dict()
)

selector_anchor_lookup = (
    selector_features_df
    .dropna(subset=['anchor_holiday_name'])
    .drop_duplicates(subset=['date'], keep='last')
    .set_index('date')['anchor_holiday_name']
    .to_dict()
)

series_unique_ids = list(UNIQUE_IDS) if 'UNIQUE_IDS' in globals() else [UNIQUE_ID]
if not series_unique_ids:
    raise ValueError('UNIQUE_IDS is empty.')
if UNIQUE_ID not in series_unique_ids:
    UNIQUE_ID = series_unique_ids[0]

def _summary_param(summary_df, param_name, default=np.nan):
    matches = summary_df.loc[summary_df['param'] == param_name, 'value']
    return matches.iloc[0] if not matches.empty else default

rolling_optuna_results = {}
rolling_runs = {}
rolling_rows = []

for unique_id in series_unique_ids:
    print(f'=== UNIQUE_ID={unique_id} ===')
    series_optuna_results = {}
    series_runs = {}

    for target_date, holiday_label in rolling_target_items:
        target_ts = pd.Timestamp(target_date).normalize()
        target_cluster = selector_cluster_lookup.get(target_ts, pd.NA)
        anchor_holiday_name = selector_anchor_lookup.get(target_ts, holiday_label)
        if pd.isna(anchor_holiday_name):
            anchor_holiday_name = holiday_label
        anchor_holiday_name = str(anchor_holiday_name)
        print(f'[{unique_id}] [{target_date}] tuning and forecasting [{anchor_holiday_name}]...')

        try:
            tuning_result = tune_analog_holidays_optuna(
                unique_id=unique_id,
                source_path=SOURCE_PATH,
                train_end=target_ts,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                initial_k=K,
                initial_typedist=TYPEDIST,
                initial_typereg=TYPEREG,
                scale_method=SCALE_METHOD,
                scale_method_choices=OPTUNA_SCALE_METHOD_CHOICES,
                initial_n_components=N_COMPONENTS,
                n_trials=OPTUNA_N_TRIALS,
                timeout_sec=OPTUNA_TIMEOUT_SEC,
                max_eval_dates=OPTUNA_MAX_EVAL_DATES,
                random_seed=OPTUNA_RANDOM_SEED,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
            )
            series_optuna_results[target_date] = tuning_result

            best_config = tuning_result.best_config
            best_k_range = tuple(best_config.get('k_range', (np.nan, np.nan)))
            best_scale_method = best_config.get('scale_method', SCALE_METHOD)
            best_regressor_params = dict(best_config.get('regressor_params', {}))
            run = run_analog_holidays(
                unique_id=unique_id,
                target_date=target_ts,
                source_path=SOURCE_PATH,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                k=int(best_config['k']),
                typedist=str(best_config['typedist']),
                typereg=str(best_config['typereg']),
                scale_method=best_scale_method,
                n_components=int(best_config['n_components']),
                regressor_params=best_regressor_params,
                levels=LEVELS,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                expected_target_label=None,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
            )
            series_runs[target_date] = run

            mae_window = np.nan
            mape_window_pct = np.nan
            if run.actual_profile is not None:
                mae_window = float(np.mean(np.abs(run.forecast_profile - run.actual_profile)))
                denom = np.where(np.abs(run.actual_profile) > 1e-9, np.abs(run.actual_profile), np.nan)
                ape_pct = np.abs(run.forecast_profile - run.actual_profile) / denom * 100.0
                if np.isfinite(ape_pct).any():
                    mape_window_pct = float(np.nanmean(ape_pct))

            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'train_end': target_date,
                'eligible_tuning_dates': len(tuning_result.eligible_dates),
                'target_exists': run.target_exists,
                'target_has_complete_profile': run.target_has_complete_profile,
                'selected_analogs': len(run.positions),
                'fail': run.fail,
                'optuna_k_min': best_k_range[0],
                'optuna_k_max': best_k_range[1],
                'k': run.k,
                'typedist': run.typedist,
                'typereg': run.typereg,
                'scale_method': run.scale_method,
                'n_components': run.n_components,
                'regressor_params': best_regressor_params,
                'forecast_start': run.forecast_start,
                'forecast_end': run.forecast_end,
                'mae_window': mae_window,
                'mape_window_pct': mape_window_pct,
                'tuning_best_mean_mae': _summary_param(tuning_result.summary_df, 'best_mean_mae'),
                'tuning_best_mean_mape_pct': _summary_param(tuning_result.summary_df, 'best_mean_mape_pct'),
                'error': None,
            })
            print(
                f'[{unique_id}] [{target_date}] cluster={target_cluster} | '
                f'k-range(optuna-param)=[{best_k_range[0]}, {best_k_range[1]}] | '
                f'k(optuna)={run.k} | '
                f'typereg(optuna)={run.typereg} | '
                f'typedist(optuna)={run.typedist} | '
                f'scale_method(optuna)={run.scale_method} | '
                f'MAPE(final)={mape_window_pct:.2f}%'
            )
        except Exception as exc:
            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'train_end': target_date,
                'eligible_tuning_dates': np.nan,
                'target_exists': False,
                'target_has_complete_profile': False,
                'selected_analogs': 0,
                'fail': True,
                'optuna_k_min': np.nan,
                'optuna_k_max': np.nan,
                'k': np.nan,
                'typedist': pd.NA,
                'typereg': pd.NA,
                'scale_method': pd.NA,
                'n_components': np.nan,
                'regressor_params': {},
                'forecast_start': pd.NaT,
                'forecast_end': pd.NaT,
                'mae_window': np.nan,
                'mape_window_pct': np.nan,
                'tuning_best_mean_mae': np.nan,
                'tuning_best_mean_mape_pct': np.nan,
                'error': str(exc),
            })
            print(f'[{unique_id}] [{target_date}] ERROR: {exc}')

    rolling_optuna_results[unique_id] = series_optuna_results
    rolling_runs[unique_id] = series_runs

rolling_daily_table = pd.DataFrame(rolling_rows)

batch_result_2025_all = {}
if not rolling_daily_table.empty:
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[['mae_window', 'mape_window_pct']].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID)
rolling_daily_table

In [ ]:
import importlib.util

current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
current_cluster_row = rolling_daily_table.loc[rolling_daily_table['target_date'] == TARGET_DATE] if 'rolling_daily_table' in globals() else pd.DataFrame()
current_cluster = current_cluster_row['analog_cluster'].iloc[0] if not current_cluster_row.empty else pd.NA
current_regressor_params = (
    dict(current_rolling_optuna_result.best_config.get('regressor_params', {}))
    if current_rolling_optuna_result is not None else {}
)
current_k_range = (
    tuple(current_rolling_optuna_result.best_config.get('k_range', (np.nan, np.nan)))
    if current_rolling_optuna_result is not None else (np.nan, np.nan)
)
current_scale_method = (
    current_rolling_optuna_result.best_config.get('scale_method', SCALE_METHOD)
    if current_rolling_optuna_result is not None else SCALE_METHOD
)

optuna_typedist_choices = ['pearson', 'euclidian']
optuna_typereg_choices = ['PCR', 'PLS']
lgbm_available = importlib.util.find_spec('lightgbm') is not None

optuna_k_rule = f'integer in [{current_k_range[0]}, {current_k_range[1]}]'
optuna_scale_method_rule = OPTUNA_SCALE_METHOD_CHOICES if OPTUNA_SCALE_METHOD_CHOICES is not None else [SCALE_METHOD]
optuna_n_components_rule = 'integer in [2, min(k, season_length)] only when typereg is PCR or PLS'
optuna_lgbm_rule = (
    'available only via explicit typereg_choices override: n_estimators in {100, 200, 300}, '
    'learning_rate in {0.03, 0.05, 0.1}, num_leaves in {15, 31, 63}, min_child_samples in {10, 20, 30}'
)
optuna_runtime_note = (
    'DTW, RidgeReg, LassoReg, RF, OLSstep, and LGBM are excluded from the default Optuna grid, '
    'and k is capped by the realizable post-filter analog pool so Optuna does not request '
    'more neighbors than the workflow can actually use.'
)

if current_rolling_optuna_result is None:
    rolling_daily_table.loc[rolling_daily_table['target_date'] == TARGET_DATE]
else:
    optuna_method_report = (
        f'OPTUNA METHOD REPORT FOR UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={TARGET_DATE}\n'
        f'- Optimization mode: single-objective minimization.\n'
        f'- Sampler: TPESampler(seed={OPTUNA_RANDOM_SEED}).\n'
        f'- Search budget: n_trials={OPTUNA_N_TRIALS}, timeout_sec={OPTUNA_TIMEOUT_SEC}.\n'
        f'- Rolling cutoff: train_end={TARGET_DATE}; only dates strictly earlier than the target are used for tuning.\n'
        f'- Backtest folds: {len(current_rolling_optuna_result.eligible_dates)} eligible historical holiday dates, capped by OPTUNA_MAX_EVAL_DATES={OPTUNA_MAX_EVAL_DATES}.\n'
        f'- Cluster restriction: match_target_cluster={MATCH_TARGET_CLUSTER}; target analog_cluster={current_cluster}.\n'
        f'- Objective function: minimize mean MAE across historical folds + 1000 * fail_rate.\n'
        f'- Search space for typedist: {optuna_typedist_choices}.\n'
        f'- Search space for typereg: {optuna_typereg_choices}.\n'
        f'- Search space for scale_method: {optuna_scale_method_rule}.\n'
        f'- Search space for k: {optuna_k_rule}.\n'
        f'- Conditional parameter for n_components: {optuna_n_components_rule}.\n'
        f'- Conditional LGBM parameters: '
        f'{optuna_lgbm_rule if lgbm_available else "not available because lightgbm is not installed"}.\n'
        f'- Runtime note: {optuna_runtime_note}\n'
        f'- Best configuration selected for this target: k(optuna)={current_rolling_optuna_result.best_config["k"]}, '
        f'typedist(optuna)={current_rolling_optuna_result.best_config["typedist"]}, '
        f'typereg(optuna)={current_rolling_optuna_result.best_config["typereg"]}, '
        f'scale_method(optuna)={current_scale_method}, '
        f'n_components(optuna)={current_rolling_optuna_result.best_config["n_components"]}, '
        f'regressor_params(optuna)={current_regressor_params}.\n'
        f'- k-range(optuna-param)=[{current_k_range[0]}, {current_k_range[1]}].'
    )
    print(optuna_method_report)
    display(current_rolling_optuna_result.summary_df)
    current_rolling_optuna_result.fold_metrics_df

In [ ]:
if (
    ('batch_result_2025_all' not in globals() or not batch_result_2025_all)
    and 'rolling_runs' in globals()
    and 'rolling_daily_table' in globals()
    and not rolling_daily_table.empty
    and 'rolling_target_items' in globals()
):
    batch_result_2025_all = {}
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[['mae_window', 'mape_window_pct']].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID) if 'batch_result_2025_all' in globals() else None
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {}
)
if not batch_results_to_plot:
    raise ValueError('Run Cell 9 first to build rolling batch results before plotting.')

batch_inference_figures = {}
batch_inference_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f'Batch inference grid | {series_unique_id}')
    fig, axes = plot_batch_inference_grid(
        series_batch_result,
        title=(
            f'Batch inference | {series_unique_id}\n'
            f'Rolling nested tuning by target date | '
            f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
        ),
    )
    batch_inference_figures[series_unique_id] = fig
    batch_inference_axes[series_unique_id] = axes
    export_figure_pdf(fig, f'batch_inference_{series_unique_id}')
    plt.show()

In [ ]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if run is None:
    run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )

summary_df = build_run_summary(run)
print(summary_df.to_string(index=False))

In [ ]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

diagnostic_run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if diagnostic_run is None:
    diagnostic_run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )

interval_rows = []
for lv in LEVELS:
    lo = diagnostic_run.interval_low.get(lv)
    hi = diagnostic_run.interval_high.get(lv)
    if lo is None or hi is None:
        continue
    width = hi - lo
    interval_rows.append({
        'level': lv,
        'average_interval_width': float(np.mean(width)),
        'max_interval_width': float(np.max(width)),
        'min_interval_width': float(np.min(width)),
    })

display(pd.DataFrame(interval_rows))

level_to_show = 95 if 95 in LEVELS else max(LEVELS)
hourly_interval_df = pd.DataFrame({
    'hour_relative_to_holiday': np.arange(len(diagnostic_run.forecast_profile)) - FORECAST_START_OFFSET_HOURS,
    'forecast_mean': diagnostic_run.forecast_profile,
    f'lower_limit_{level_to_show}': diagnostic_run.interval_low[level_to_show],
    f'upper_limit_{level_to_show}': diagnostic_run.interval_high[level_to_show],
})

hourly_interval_df.head(10)

### X/X2 and Y/Y2 Sequences — Batch Grid

Una gráfica por fecha pronosticada, mostrando los pares históricos X/X2 en azul claro y la secuencia Y/Y2 forecast en rojo.

En esta variante la ventana Y2/X2 tiene 38 horas y comienza 14 horas antes del inicio del holiday.

In [ ]:
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {UNIQUE_ID: batch_result_2025}
    if 'batch_result_2025' in globals() and batch_result_2025 is not None else {}
 )
if not batch_results_to_plot:
    raise ValueError('No batch results are available to plot.')

batch_pair_figures = {}
batch_pair_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f'X/X2 and Y/Y2 grid | {series_unique_id}')
    fig_seq, axes_seq = plot_batch_pair_sequences_grid(
        series_batch_result,
        title=(
            f'X/X2 y Y/Y2 por fecha pronosticada | {series_unique_id}\n'
            f'Rolling nested tuning por fecha | '
            f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
        ),
    )
    batch_pair_figures[series_unique_id] = fig_seq
    batch_pair_axes[series_unique_id] = axes_seq
    export_figure_pdf(fig_seq, f'batch_pair_sequences_{series_unique_id}')
    plt.show()

In [ ]:
ranking_df = build_analog_ranking_table(run)
fig, axes = plot_ranked_analog_profiles(run, top_n=MAX_PLOTTED_ANALOGS)
export_figure_pdf(fig, f'ranked_analog_profiles_{UNIQUE_ID}_{TARGET_DATE}')
plt.show()
ranking_df.head(MAX_PLOTTED_ANALOGS)